# 05 — Implied Volatility

I invert the pricing equation with three solvers because the numerical behavior is as important as the final volatility. The low-vega examples show where the inversion becomes fragile.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
from options_lab.black_scholes import black_scholes_price
from options_lab.implied_volatility import implied_volatility_bisection, implied_volatility_newton, implied_volatility_brent

true_vol=.20
price=black_scholes_price(100.,100.,.03,1.,true_vol,"call")
results=[]
for name,solver in [("bisection",implied_volatility_bisection),("newton",implied_volatility_newton),("brent",implied_volatility_brent)]:
    kwargs=dict(market_price=price,spot=100.,strike=100.,rate=.03,time_to_expiry=1.,option_type="call")
    if name=="newton": kwargs["initial_guess"]=.30
    r=solver(**kwargs)
    results.append({"method":name,"iv":r.volatility,"iterations":r.iterations,"residual":r.residual,"converged":r.converged})
pd.DataFrame(results)

,method,iv,iterations,residual,converged
0,bisection,0.2,34,-6.752074e-10,True
1,newton,0.2,4,0.000000e+00,True
2,brent,0.2,6,0.000000e+00,True


In [3]:
from options_lab.greeks import black_scholes_greeks
cases=[("ATM",100.,100.,30/365), ("Deep OTM",100.,150.,30/365), ("Very short OTM",100.,120.,2/365)]
rows=[]
for label,s,k,t in cases:
    p=black_scholes_price(s,k,.03,t,.20,"call")
    v=black_scholes_greeks(s,k,.03,t,.20,"call").vega
    rows.append({"case":label,"price":p,"raw_vega":v})
pd.DataFrame(rows)

,case,price,raw_vega
0,ATM,2.409581e+00,1.140798e+01
1,Deep OTM,1.003102e-12,2.622684e-10
2,Very short OTM,5.604709e-36,4.325474e-33


**What I took from it.** Implied volatility is well-conditioned when vega is substantial. In the low-vega wings, a small price error can become a large volatility error.